In [ ]:
# Google Colab setup
!pip install -q polars plotly gdown python-dotenv kailash-ml kailash-kaizen

# Mount Google Drive for datasets
from google.colab import drive
drive.mount("/content/drive")


# ════════════════════════════════════════════════════════════════════════
# ASCENT05 — Exercise 3: ReAct Agents with Tools
# ════════════════════════════════════════════════════════════════════════
# OBJECTIVE: Build a ReActAgent with custom tools for autonomous data
#   exploration. Observe reasoning-action traces and tool selection.
#
# TASKS:
#   1. Define custom tools (DataExplorer, FeatureEngineer, ModelVisualizer)
#   2. Build ReActAgent with tool access
#   3. Run autonomous data exploration
#   4. Observe and analyse reasoning-action trace
#   5. Safety: what happens without cost budgets?
# ════════════════════════════════════════════════════════════════════════


In [ ]:
# Copyright 2026 Terrene Foundation
# SPDX-License-Identifier: Apache-2.0
from __future__ import annotations

import os

import polars as pl
from dotenv import load_dotenv

from kaizen import Signature, InputField, OutputField
from kaizen_agents.agents.specialized.react import ReActAgent

from shared import ASCENTDataLoader
from shared.kailash_helpers import setup_environment

setup_environment()

model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
if not model or not os.environ.get("OPENAI_API_KEY"):
    print("Set OPENAI_API_KEY and DEFAULT_LLM_MODEL in .env to run this exercise")
    raise SystemExit(0)



### Data Loading


In [ ]:
loader = ASCENTDataLoader()
credit = loader.load("ascent03", "sg_credit_scoring.parquet")



## TASK 1: Define tools for the ReActAgent


In [ ]:
# Tools are functions the agent can call during reasoning
async def tool_profile_data(dataset_name: str) -> str:
    """Profile a dataset using DataExplorer and return summary."""
    # TODO: Import DataExplorer from kailash_ml; create an explorer and
    #   await explorer.profile() on a 10_000-row sample of credit (seed=42)
    # Hint: from kailash_ml import DataExplorer
    #       explorer = DataExplorer()
    #       profile = await explorer.profile(credit.sample(n=min(10_000, credit.height), seed=42))
    # TODO: Return formatted string with dataset_name, profile.n_rows, profile.n_columns,
    #   len(profile.alerts), profile.type_summary, and top 5 alert types+columns
    ____


async def tool_check_correlations(threshold: float = 0.8) -> str:
    """Find highly correlated features."""
    # TODO: Import DataExplorer and AlertConfig; create explorer with
    #   AlertConfig(high_correlation_threshold=threshold)
    # Hint: from kailash_ml import DataExplorer
    #       from kailash_ml.engines.data_explorer import AlertConfig
    # TODO: Profile a 5000-row sample; iterate profile.correlation_matrix
    #   to find pairs where abs(corr) > threshold (skip self, skip duplicates)
    # Hint: profile.correlation_matrix is dict[str, dict[str, float]]
    # TODO: Return formatted string listing high-correlation pairs (up to 10)
    ____


def tool_describe_column(column_name: str) -> str:
    """Get statistics for a specific column."""
    # TODO: Return an error string if column_name not in credit.columns
    # TODO: For numeric types (Float64, Float32, Int64, Int32): return mean, std, min, max, nulls
    # Hint: col.dtype in (pl.Float64, pl.Float32, pl.Int64, pl.Int32)
    # TODO: For categorical: return n_unique, null_count, and top 5 value_counts
    ____


def tool_default_rate_by(column_name: str) -> str:
    """Compute default rate grouped by a column."""
    # TODO: Return error string if column_name not in credit.columns
    # TODO: Group credit by column_name; aggregate default mean and count;
    #   sort descending by default_rate; take top 10
    # Hint: credit.group_by(column_name).agg(
    #           pl.col("default").mean().alias("default_rate"),
    #           pl.col("default").count().alias("n"),
    #       ).sort("default_rate", descending=True).head(10)
    # TODO: Format rows as "  {value}: {rate:.3f} (n={n})" and return joined string
    ____


# Tool registry
tools = {
    "profile_data": tool_profile_data,
    "check_correlations": tool_check_correlations,
    "describe_column": tool_describe_column,
    "default_rate_by": tool_default_rate_by,
}



## TASK 2: Build ReActAgent


In [ ]:
# TODO: Define DataExplorationResult as a Signature subclass
#   - InputField: task (str, data exploration task)
#   - OutputField: findings (list[str], key findings from exploration)
#   - OutputField: tools_used (list[str], tools invoked during exploration)
#   - OutputField: recommendation (str, recommendation based on findings)
____  # Hint: class DataExplorationResult(Signature): ...


def run_react_agent():
    # TODO: Create a ReActAgent with model
    agent = ____  # Hint: ReActAgent(model=model)

    task = (
        "Explore the Singapore credit scoring dataset. "
        "Profile the data, check for high correlations, "
        "find which features most strongly predict default, "
        "and recommend preprocessing steps for a classification model."
    )

    # Build context from the available tool functions so the agent
    # knows what data operations are available
    tool_descriptions = "\n".join(
        f"  {name}: {func.__doc__ or 'no description'}" for name, func in tools.items()
    )
    context = (
        f"Available analysis functions:\n{tool_descriptions}\n\n"
        f"Dataset: sg_credit_scoring.parquet with {credit.height:,} rows "
        f"and columns: {credit.columns}"
    )

    print(f"\n=== ReActAgent Exploration ===")
    print(f"Task: {task}")
    print(f"Available tools: {list(tools.keys())}")
    print(f"Budget: governed by Delegate-level budget_usd\n")

    # TODO: Run the agent passing task and context keyword arguments
    result = ____  # Hint: agent.run(task=task, context=context)

    print(f"\n=== Results ===")
    print(f"Result keys: {list(result.keys())}")
    for key, value in result.items():
        print(f"  {key}: {str(value)[:200]}")

    return result


react_result = run_react_agent()



## TASK 3: Analyse reasoning-action trace


In [ ]:
print(f"\n=== Reasoning-Action Trace ===")
print("ReAct loop: Thought → Action → Observation → Thought → ...")
print(f"The agent autonomously decided which tools to call and in what order.")
print(f"This is the key difference from CoT (Exercise 2):")
print(f"  CoT: reason step-by-step, produce answer")
print(f"  ReAct: reason, ACT on the world (call tools), observe results, reason again")



## TASK 4: Safety — cost budgets and runaway agents


In [ ]:
print(f"\n=== Safety: Cost Budgets ===")
print(
    """
What happens if you REMOVE the cost budget?

1. The agent could loop indefinitely (calling tools → reasoning → more tools)
2. Each LLM call costs money — 100 iterations at $0.05/call = $5.00
3. What if the agent calls DataExplorer on a 100GB dataset?
   → Memory exhaustion, cluster costs, timeout failures

budget_usd is NOT optional. It is a governance requirement.

In production:
  - Set budgets proportional to task complexity
  - Monitor actual spend vs budget
  - PACT (Module 6) enforces budgets as operating envelopes
  - Agents cannot modify their own budget (frozen GovernanceContext)
"""
)

print("\n✓ Exercise 3 complete — ReActAgent with tools and safety analysis")

